# Orislop web → Hugging Face ingest

No Google Drive is used. Worker 0 discovers or loads a rights-reviewed catalog, creates a bounded four-worker plan, and publishes it to a private Hugging Face dataset. Each runtime uses ephemeral disk, uploads its assigned batches, verifies them, writes completion markers, and deletes local media.

Social APIs may provide metadata/reference records only. Raw media must come from official open-media APIs, licensed Hub files, or authorized direct non-platform endpoints. Quarantine mode collects review candidates without turning provisional labels into training ground truth.

In [ ]:
WORKER_ID = 0  # use 0, 1, 2, or 3 in separate runtimes
CREATE_PLAN = True  # true only for worker 0, once
DISCOVER_FRESH = True  # worker 0 queries Wikimedia's official API
MODE = 'quarantine'  # use 'production' only for fully reviewed rows
HF_REPO_ID = 'gonnerthetooner/orislop-web-corpus'
HF_REVISION = 'orislop-ingest-v1'
RUN_ID = 'orislop-web-v1'
REPO_URL = 'https://github.com/coolguy860/Orislop-landing.git'
REPO_BRANCH = 'main'
MAX_BATCH_GIB = 50
DOWNLOAD_THREADS = 4

In [ ]:
from google.colab import userdata
from pathlib import Path
import os, shutil, subprocess
token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Add a Hugging Face write token as the HF_TOKEN Colab secret')
os.environ['HF_TOKEN'] = token
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
repo = Path('/content/orislop')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, str(repo)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(repo / 'training/orislop_dataset/requirements.txt')], check=True)
ingest = repo / 'training/orislop_dataset/hf_web_ingest.py'
discover = repo / 'training/orislop_dataset/discover_sources.py'
if not ingest.is_file():
    raise FileNotFoundError('hf_web_ingest.py is not on the selected GitHub branch; push the current training/ changes first')

In [ ]:
plan_root = Path('/content/orislop-hf-plan')
catalog = Path('/content/orislop-web-candidates.jsonl')
if CREATE_PLAN:
    if WORKER_ID != 0:
        raise RuntimeError('Only worker 0 may create and publish the plan')
    if plan_root.exists():
        shutil.rmtree(plan_root)
    if DISCOVER_FRESH:
        subprocess.run([
            'python', str(discover), 'wikimedia',
            '--category', 'Videos of interviews',
            '--category', 'Videos of speeches',
            '--category', 'Videos of lectures',
            '--limit-per-category', '5000',
            '--output', str(catalog),
        ], cwd=repo, check=True)
    else:
        catalog = repo / 'training/orislop_dataset/catalogs/wikimedia_discovery_seed.jsonl'
    subprocess.run([
        'python', str(ingest), 'plan', '--catalog', str(catalog),
        '--output-root', str(plan_root), '--repo-id', HF_REPO_ID,
        '--revision', HF_REVISION, '--run-id', RUN_ID, '--mode', MODE,
        '--workers', '4', '--max-batch-gib', str(MAX_BATCH_GIB), '--max-files', '80',
    ], cwd=repo, check=True)
    subprocess.run(['python', str(ingest), 'publish-plan', '--plan-root', str(plan_root)], cwd=repo, check=True)
else:
    if plan_root.exists():
        shutil.rmtree(plan_root)
    subprocess.run([
        'python', str(ingest), 'fetch-plan', '--repo-id', HF_REPO_ID,
        '--revision', HF_REVISION, '--run-id', RUN_ID, '--output-root', str(plan_root),
    ], cwd=repo, check=True)

In [ ]:
# Run in each runtime after worker 0 has published the plan.
subprocess.run([
    'python', str(ingest), 'run-worker', '--plan-root', str(plan_root),
    '--worker-id', str(WORKER_ID), '--stage-root', '/content/orislop-hf-stage',
    '--download-threads', str(DOWNLOAD_THREADS),
], cwd=repo, check=True)

In [ ]:
# Check progress at any time. Finalize only after all four workers finish.
status = subprocess.run(['python', str(ingest), 'status', '--plan-root', str(plan_root)], cwd=repo)
if WORKER_ID == 0 and status.returncode == 0:
    subprocess.run(['python', str(ingest), 'finalize', '--plan-root', str(plan_root)], cwd=repo, check=True)